---
**Домашнее задание №5 (с проверкой преподавателя).**
---


---
**Выполнил: Юсов А.А.**
---

Описание задания:

Ваша задача — реализовать класс Account, который моделирует поведение банковского счёта. Этот класс должен не только выполнять базовые операции, но и вести детальный учёт всех действий, а также предоставлять аналитику по истории операций.

**Этап 1. Реализация базового класса Account**

Класс должен быть инициализирован с параметрами:

* account_holder (str): имя владельца счёта;

* balance (float, по умолчанию 0): начальный баланс счёта, не может быть отрицательным.

Атрибуты:

* holder: хранит имя владельца;

* _balance: приватный атрибут для хранения текущего баланса;

* operations_history: список или другая структура для хранения истории операций.

**Важно:** каждая операция должна храниться не просто как число, а как структурированная информация, например, словарь или кортеж. Минимальный набор данных для операции: тип операции ('deposit' или 'withdraw'), сумма, дата и время операции, текущий баланс после операции, статус ('success' или 'fail').

**Этап 2. Реалирзация методов**

1) __init__(self, account_holder, balance=0): конструктор;

2) Deposit(self, amount): метод для пополнения счёта:

* принимает сумму (должна быть положительной);

* в случае успеха обновляет баланс и добавляет запись в историю операций.

3) withdraw(self, amount): метод для снятия средств:

* принимает сумму (должна быть положительной);
* проверяет, достаточно ли средств на счёте, если нет — операция не проходит, но ее попытка с статусом 'fail' все равно фиксируется в истории;
* в случае успеха обновляет баланс и добавляет запись.

4) get_balance(self): метод, который возвращает текущий баланс.

5) get_history(self): метод, который возвращает историю операций.

**Важно:** продумайте, в каком формате его вернуть. Для работы с датой и временем используйте модуль datetime. Получить текущее время можно с помощью datetime.now().

**Этап 3. Реализация наследования.**

Создайте класс CreditAccount(Account), который наследует всю функциональность класса Account и добавляет новую.

Особенности кредитного счёта:

1) При инициализации принимает дополнительный параметр credit_limit (кредитный лимит). Баланс такого счета может быть отрицательным, но не ниже значения -credit_limit.

2) По запросу показывает, сколько кредитных средств еще доступно (текущий баланс + кредитный лимит).

3) В историю операций добавляется информацию о том, были ли использованы кредитные средства в данной операции.

In [ ]:
from datetime import datetime

log_file = None  # пока none!!!
# Написал такую логику для перезаписи файла при повторных запусках программы (чтобы не было ошибки)
# Хотя как вариант можно сделать без перезаписи но с добавлением логов при каждом запуске (но в таком случае файл будет разрастаться)
# Но так как это дополнение (не входит в задание) остановился на варианте перезаписи файла


def log_open(filename='account_log.txt'):
    """
    Открытие (перезапись) файла для логов

    Args:
        filename(str): Имя файла для логов. По умолчанию 'account_log.txt'
    """
    global log_file
    if log_file is not None and not log_file.closed:
        log_file.close()
    log_file = open(filename, 'w', encoding='utf-8')


def print_and_log(message=''):
    """Печать сообщения в консоль и запись в файл лога (сообщение для вывода, но по умолчанию пустая строка)"""
    # Как писал выше -  решил сделать с записью в файл для логов (говорили на одном из вебинаров что можно дополнить по желанию чем то своим)
    print(message)
    if log_file is not None and not log_file.closed:
        log_file.write(message + '\n')


class Account:
    """
    Банковский счет с историей операций

    Attributes:
        holder (str): Имя владельца счета
        _balance (float): Текущий баланс (приватный)
        operations_history (list): Список словарей с историей операций
    """

    def __init__(self, account_holder, balance=0):
        """
        Инициализация банковского счета

        Args:
            account_holder (str): Имя владельца счета
            balance (float): Начальный баланс. По умолчанию 0
                             Если отрицательный — устанавливается 0

        Raises:
            ValueError: Если account_holder не является строкой
        """
        if not isinstance(account_holder, str):
            raise ValueError('Имя владельца должно быть строкой')

        self.holder = account_holder
        # В задании указан тип float, но в реальной задаче я бы написал чуть иначе!!!
        # Вместо float я бы использовал Decimal (точность которая в сфере финансов очень важна), но по заданию float...
        self._balance = balance if balance >= 0 else 0
        self.operations_history = []

    @property
    def balance(self):
        """Геттер для приватного атрибута _balance (возврат текущего счета баланса)"""
        return self._balance

    @staticmethod
    def is_valid_amount(amount):
        """
        Проверка, что сумма это число и оно больше нуля

        Args:
            amount: Проверяемое значение

        Returns:
            bool: True, если amount — положительное число, иначе False
        """
        if not isinstance(amount, (int, float)):
            return False
        return amount > 0

    def deposit(self, amount):
        """
        Пополнение счета на указанную сумму.

        Если сумма невалидна (не число или <= 0), операция фиксируется
        в истории со статусом fail, но баланс не меняется

        Args:
            amount (float): Сумма пополнения (должна быть > 0)
        """
        if not self.is_valid_amount(amount):
            self._log_operation('deposit', amount, 'fail')
            print_and_log(f'[!] Ошибка: сумма должна быть положительным числом (получено {amount})')
            return

        self._balance += amount
        self._log_operation('deposit', amount, 'success')
        print_and_log(f'[+] Пополнение на {amount:.2f} | баланс: {self._balance:.2f}')

    def withdraw(self, amount):
        """
        Снятие средств со счета

        Если сумма невалидна или превышает баланс, операция фиксируется
        в истории со статусом fail, но баланс не меняется

        Args:
            amount (float): Сумма снятия (должна быть > 0 и <= баланса)
        """
        if not self.is_valid_amount(amount):
            self._log_operation('withdraw', amount, 'fail')
            print_and_log(f'[!] Ошибка: сумма должна быть положительным числом (получено {amount})')
            return

        if amount > self._balance:
            self._log_operation('withdraw', amount, 'fail')
            print_and_log(f'[!] Ошибка: недостаточно средств (запрошено {amount}, доступно {self._balance})')
            return

        self._balance -= amount
        self._log_operation('withdraw', amount, 'success')
        print_and_log(f'[-] Снятие {amount:.2f} | баланс: {self._balance:.2f}')

    def _log_operation(self, op_type, amount, status):
        """
        Внутренний метод (запись операции в историю)

        Args:
            op_type (str): Тип операции ('deposit' или 'withdraw')
            amount: Сумма операции (число или строка)
            status (str): Статус ('success' или 'fail')
        """
        safe_amount = amount if isinstance(amount, (int, float)) else 0

        self.operations_history.append({
            'type': op_type,
            'amount': safe_amount,
            'date': datetime.now(),
            'balance_after': self._balance,
            'status': status
        })

    def get_balance(self):
        """Возврат текущего баланса счета"""
        return self._balance

    def get_history(self):
        """Возврат истории операций"""
        if not self.operations_history:
            return 'История операций пуста'

        lines = []
        for i, op in enumerate(self.operations_history, 1):
            date_str = op['date'].strftime('%Y-%m-%d %H:%M:%S')
            status_mark = 'OK  ' if op['status'] == 'success' else 'FAIL'

            if op['status'] == 'success':
                sign = '+' if op['type'] == 'deposit' else '-'
                amount_str = f"{sign}{op['amount']:>8.2f}"
            else:
                amount_str = f"{str(op['amount']):>9}"

            lines.append(
                f"{i:>2}. [{status_mark}] {date_str} | {amount_str} | "
                f"баланс: {op['balance_after']:>10.2f}"
            )
        return '\n'.join(lines)

    @classmethod
    def from_dict(cls, data):
        """
        Фабрика (создание счета из словаря)

        Args:
            data (dict): Словарь с ключами holder (обязательно) и balance (опционально)

        Returns:
            Account: Новый экземпляр счета

        Raises:
            ValueError: Если в словаре нет ключа holder или неверные данные
        """
        try:
            return cls(
                account_holder=data['holder'],
                balance=data.get('balance', 0)
            )
        except KeyError as e:
            raise ValueError(f'В словаре нет ключа {e.args[0]}')
        except (TypeError, ValueError) as e:
            raise ValueError(f'Неверные данные для счета: {e.args[0]}')

    def __str__(self):
        """Строковое представление счета для пользователя"""
        return f'Счет «{self.holder}» | баланс: {self._balance:.2f}'

    def __repr__(self):
        """Строковое представление счета для отладки"""
        return f'Account({self.holder!r}, {self._balance})'


class CreditAccount(Account):
    """
    Кредитный счет (баланс может уходить в минус до определенного -credit_limit)

    Attributes:
        credit_limit (float): Максимальный кредитный лимит
    """

    def __init__(self, account_holder, balance=0, credit_limit=0):
        """
        Инициализация кредитного счета

        Args:
            account_holder (str): Имя владельца счета
            balance (float): Начальный баланс. По умолчанию 0
            credit_limit (float): Кредитный лимит. По умолчанию 0
        """
        super().__init__(account_holder, balance)
        self.credit_limit = credit_limit

    def withdraw(self, amount):
        """
        Снятие средств с учетом кредитного лимита

        Баланс может уходить в минус, но не ниже -credit_limit
        Соответственно если операция превышает лимит, то она фиксируется со статусом 'fail'

        Args:
            amount (float): Сумма снятия (должна быть > 0)
        """
        if not self.is_valid_amount(amount):
            self._log_operation('withdraw', amount, 'fail', credit_used=False)
            print_and_log(f'[!] Ошибка: сумма должна быть положительным числом (получено {amount})')
            return

        if self._balance - amount < -self.credit_limit:
            self._log_operation('withdraw', amount, 'fail', credit_used=False)
            print_and_log(f'[!] Ошибка: превышен кредитный лимит '
                          f'(доступно {self._balance + self.credit_limit:.2f})')
            return

        self._balance -= amount
        credit_used = self._balance < 0
        self._log_operation('withdraw', amount, 'success', credit_used=credit_used)
        credit_note = ' (за счет кредита)' if credit_used else ''
        print_and_log(f'[-] Снятие {amount:.2f}{credit_note} | баланс: {self._balance:.2f}')

    def _log_operation(self, op_type, amount, status, credit_used=False):
        """
        Переопределение (добавление поля credit_used в историю)

        Args:
            op_type (str): Тип операции
            amount: Сумма операции
            status (str): Статус ('success' или 'fail')
            credit_used (bool): Использованы ли кредитные средства
        """
        # если amount не число - записываем как 0 для истории (или 'N/A')
        safe_amount = amount if isinstance(amount, (int, float)) else 0

        self.operations_history.append({
            'type': op_type,
            'amount': safe_amount,
            'date': datetime.now(),
            'balance_after': self._balance,
            'status': status,
            'credit_used': credit_used
        })

    def get_available_credit(self):
        """Возврат доступных кредитных средств"""
        return self._balance + self.credit_limit

    def get_history(self):
        """История операций с пометкой кредитных операций"""
        if not self.operations_history:
            return 'История операций пуста'

        lines = []
        for i, op in enumerate(self.operations_history, 1):
            date_str = op['date'].strftime('%Y-%m-%d %H:%M:%S')
            status_mark = 'OK  ' if op['status'] == 'success' else 'FAIL'
            credit_mark = ' [КРЕДИТ]' if op.get('credit_used') else ''

            if op['status'] == 'success':
                sign = '+' if op['type'] == 'deposit' else '-'
                amount_str = f"{sign}{op['amount']:>8.2f}"
            else:
                amount_str = f"{str(op['amount']):>9}"

            lines.append(
                f"{i:>2}. [{status_mark}] {date_str} | {amount_str} | "
                f"баланс: {op['balance_after']:>10.2f}{credit_mark}"
            )
        return '\n'.join(lines)

    @classmethod
    def from_dict(cls, data):
        """
        Фабрика для кредитного счета

        Args:
            data (dict): Словарь с ключами holder, balance, credit_limit'

        Returns:
            CreditAccount: Новый экземпляр кредитного счета

        Raises:
            ValueError: Если в словаре нет ключа 'holder' или неверные данные
        """
        try:
            return cls(
                account_holder=data['holder'],
                balance=data.get('balance', 0),
                credit_limit=data.get('credit_limit', 0)
            )
        except KeyError as e:
            raise ValueError(f'В словаре нет ключа {e.args[0]}')
        except (TypeError, ValueError) as e:
            raise ValueError(f'Неверные данные для кредитного счета: {e.args[0]}')

    def __str__(self):
        """Строковое представление кредитного счета для пользователя"""
        return (f'Кредитный счет «{self.holder}» | баланс: {self._balance:.2f} | '
                f'лимит: {self.credit_limit:.2f} | доступно: {self.get_available_credit():.2f}')

Ниже прописал всю необходимую документацию

In [ ]:
# Документация (help) по объектам классов
print('\n' + '=' * 60)
print('  ДОКУМЕНТАЦИЯ (help)')
print('=' * 60)

help(acc)
help(acc2)
help(credit)
help(credit2)

Ну и сам запуск программы (сделал отдельным блоком чтобы "не резало" глаз)

Добавил также проверку обработки ошибок (для наглядности)!

In [ ]:
log_open()                           # открытие (перезапись) файла account_log.txt для логов
print_and_log('=' * 60)              # печать разделителя в консоль и файл
print_and_log('  БАНКОВСКИЕ СЧЕТА')  # заголовок раздела
print_and_log('=' * 60)              # разделитель

acc = Account('Иван', 1000) # создание дебетового счета Ивана с балансом 1000
acc.deposit(500)            # пополнение на 500 (баланс 1500)
acc.withdraw(200)           # снятие 200 (баланс 1300)
acc.withdraw(5000)          # попытка снять 5000 (fail недостаток средств)
acc.deposit(-100)           # попытка пополнить на -100 (fail отрицательная сумма)
acc.deposit('abc')          # попытка пополнить на строку "abc" (очередной fail так как не число)

print_and_log()                            # пустая строка (радость для глаз)
print_and_log(str(acc))                    # вывод счета через __str__  (Счет «Иван» | баланс 1300)
print_and_log('--- История операций ---')  # заголовок истории
print_and_log(acc.get_history())           # вывод всей истории операций

print_and_log()                    # пустая строка
print_and_log('=' * 60)            # разделитель
print_and_log('  КРЕДИТНЫЙ СЧЕТ')  # заголовок раздела
print_and_log('=' * 60)            # разделитель

credit = CreditAccount('Мария', 100, credit_limit=1000)  # создание кредитного счета Марии (баланс 100, лимит 1000)
credit.withdraw(500)                                     # снятие 500 (баланс -400 за счет кредита)
credit.withdraw(2000)                                    # попытка снять 2000 (fail превышен лимит)

print_and_log()                            # пустая строка
print_and_log(str(credit))                 # вывод кредитного счета через __str__
print_and_log('--- История операций ---')  # заголовок истории
print_and_log(credit.get_history())        # вывод истории с пометкой [КРЕДИТ]

print_and_log()                         # пустая строка
print_and_log('=' * 60)                 # разделитель
print_and_log('  СОЗДАНИЕ ИЗ СЛОВАРЯ')  # заголовок раздела
print_and_log('=' * 60)                 # разделитель

acc2 = Account.from_dict({'holder': 'Петр', 'balance': 500})  # создание счета через фабрику from_dict
print_and_log(str(acc2))                                      # вывод счета Петра

credit2 = CreditAccount.from_dict({'holder': 'Ольга', 'balance': 0, 'credit_limit': 500})  # создание кредитного счета через фабрику
print_and_log(str(credit2))                                                                # вывод кредитного счета Ольги

try:                                           # попытка создать счет из неполного словаря
    bad = Account.from_dict({'balance': 100})  # нет ключа holder (ValueError)
except ValueError as e:                        # !!!здесь ловим ошибку!!!
    print_and_log(f'[!] Ошибка: {e.args[0]}')  # вывод сообщения об ошибке

log_file.close()                               # клоузаем файл в конце
print('\n[OK] Лог сохранен в account_log.txt') # сообщение только в консоль (не в файл)